# Image Upscale — SeedVR2 — ComfyUI

SeedVR2 7B modeli ile görsel upscale.

**Model:** SeedVR2 7B

**Custom Nodes:** Yok (saf ComfyUI core)

In [ ]:
# ===== Section A: ComfyUI Kurulumu =====
import os

COMFY_DIR = '/content/ComfyUI'

if not os.path.exists(COMFY_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    %cd {COMFY_DIR}
    !pip install -r requirements.txt
    # ComfyUI Manager
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager
else:
    %cd {COMFY_DIR}
    !git pull
    print('ComfyUI zaten kurulu.')

In [ ]:
# ===== Section B: Custom Nodes =====
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'
NODES = {
    'ComfyUI_essentials': 'https://github.com/cubiq/ComfyUI_essentials.git',
    'ComfyUI_LayerStyle': 'https://github.com/chflame163/ComfyUI_LayerStyle.git',
    'ComfyUI-Impact-Pack': 'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git',
    'ComfyUI-SeedVR2_VideoUpscaler': 'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git',
}
for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

In [ ]:
# ===== Section B2: Model İndirme =====
import os

models_dir = f'{COMFY_DIR}/models'

# SeedVR2 7B model from huggingface.co/Skywork/SeedVR2 - TODO: verify exact file
# os.makedirs(f'{models_dir}/upscale_models', exist_ok=True)
# !wget -c -O {models_dir}/upscale_models/seedvr2_7b.safetensors \
#     'https://huggingface.co/Skywork/SeedVR2/resolve/main/seedvr2_7b.safetensors'

print('⚠️ Model dosya adını doğrulayıp indirme satırını aktif edin.')

In [ ]:
# ===== Section C: ComfyUI Başlatma =====
import subprocess
import time

import requests
from google.colab import userdata, output

USE_CLOUDFLARE = False

PORT = 8188

subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*', '--enable-manager'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI başlatıldı (PID: {comfy_proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI çöktü!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI hazır ({int(time.time()-t0)}s)')
    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
# ===== Section D: Workflow Yükleme =====
import json

# TODO: Workflow JSON'unu buraya ekleyin
workflow = {}

print('Workflow hazır. Section E ile çalıştırabilirsiniz.')

In [ ]:
# ===== Section E: Workflow Çalıştırma =====
import json

import requests

def queue_prompt(prompt_workflow, server='127.0.0.1'):
    url = f'http://{server}:{PORT}/prompt'
    payload = {'prompt': prompt_workflow}
    resp = requests.post(url, json=payload)
    if resp.status_code == 200:
        print(f'✅ Workflow kuyruğa eklendi: {resp.json()["prompt_id"]}')
    else:
        print(f'❌ Hata: {resp.status_code} — {resp.text[:200]}')

queue_prompt(workflow)

In [ ]:
# ===== Section F: Keepalive =====
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canlı tutma. Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)